# Service Reminder Analytics Pipeline — Automotive Aftersales

Portfolio version of an operational service-reminder workflow developed during a **Business Data Analyst internship at Auto2000 Kenjeran**.

The pipeline combines historical service data, business-rule scheduling, machine-learning risk scoring, Google Sheets output, WhatsApp reminder-link generation, and Power BI monitoring.

## Workflow

**Historical PKB & SO Data → Cleaning & Integration → Service Eligibility Rules → Reminder Scheduling → XGBoost Risk Scoring → Reminder Target List → Google Sheets → Power BI**

The machine-learning model is used to **prioritize reminder targets through a risk score**, while service timing and eligibility are governed by operational business rules.

> **Portfolio note:** Customer-level records, phone numbers, Google Drive paths, spreadsheet URLs, credentials, branch contact details, trained model artifacts, and internal source files are intentionally excluded from this public version.

---


### **SETUP DRIVE, PATH, & KONFIG OPERASIONAL**

#### Bulan target operasional (ubah target bulan yang ingin di reminder)

In [ ]:
# Bulan target operasional (ubah target bulan yang ingin di reminder)
TARGET_MONTH = "YYYY-MM"  # set operational target month     # format "YYYY-MM"

*(Bagian ini mengatur lokasi file, bulan target, dan nama output)*

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, re, json, joblib
import pandas as pd
import numpy as np
from urllib.parse import quote

BASE_DIR   = "/content/drive/MyDrive/service_reminder_project"
MODEL_PATH = f"{BASE_DIR}/Model/pipe_xgb.joblib"
META_PATH  = f"{BASE_DIR}/Model/model_features.json"

MASTER_DIR = f"{BASE_DIR}/Data History"
OUT_DIR    = f"{BASE_DIR}/Output"

os.makedirs(MASTER_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

PKB_PATH = f"{MASTER_DIR}/PKB.xlsx"
SO_PATH  = f"{MASTER_DIR}/SO.xlsx"

# Nama Google Sheet + worksheet
GSHEET_NAME = "Reminder_Output"
WS_FACT     = "All Customer"
WS_ACTIVE   = "Reminder_Target"

print("BASE_DIR:", BASE_DIR)
print("TARGET_MONTH:", TARGET_MONTH)

### **LOAD MODEL & META FITUR**

*(Model digunakan untuk memberikan skor risiko untuk urutan reminder)*

In [ ]:
def load_model_and_meta(model_path: str, meta_path: str):
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model tidak ditemukan: {model_path}")
    if not os.path.exists(meta_path):
        raise FileNotFoundError(f"Meta tidak ditemukan : {meta_path}")

    pipe = joblib.load(model_path)
    with open(meta_path, "r") as f:
        meta = json.load(f)

    num_feats = meta.get("num_feats", [])
    cat_feats = meta.get("cat_feats", [])
    return pipe, num_feats, cat_feats

pipe_xgb, num_feats, cat_feats = load_model_and_meta(MODEL_PATH, META_PATH)

print("Loaded model:", MODEL_PATH)

### **DATA CLEANING (PKB & SO)**

*Membuat nama kolom konsisten*

In [ ]:
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = (out.columns.astype(str)
                   .str.strip()
                   .str.replace(r"\s+", " ", regex=True))
    return out

def clean_pkb(df: pd.DataFrame) -> pd.DataFrame:
    out = standardize_columns(df)

    out = out.rename(columns={
        "Service Category": "ServiceCategory",
        "Service Advisor": "ServiceAdvisor",
        "Estimated Revenue": "EstimatedRevenue",
        "Contact Person": "ContactPerson",
        "Contact Person Phone": "ContactPersonPhone",
        "Customer Name": "CustomerName",
        "Police Reg No": "PoliceRegNo",
        "PoliceRegNo": "PoliceRegNo",
    })

    required = ["PKBNo","PKBDate","EquipmentNo","PKBType","ServiceCategory"]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"Kolom wajib PKB tidak ada: {missing}")

    out["PKBNo"] = out["PKBNo"].astype(str).str.strip()
    out["EquipmentNo"] = out["EquipmentNo"].astype(str).str.strip().str.upper()
    out["PKBDate"] = pd.to_datetime(out["PKBDate"], errors="coerce", dayfirst=True)
    out = out.dropna(subset=["PKBNo","EquipmentNo","PKBDate"]).copy()

    for col in ["ServiceCategory","PKBType","TypeKendaraan","ServiceAdvisor"]:
        if col in out.columns:
            out[col] = out[col].fillna("UNKNOWN").astype(str).str.strip().str.upper()

    for col in ["Kilometer", "EstimatedRevenue"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    return out

def clean_so(df: pd.DataFrame) -> pd.DataFrame:
    out = standardize_columns(df)

    equip_candidates = ["Equipment No", "EquipmentNo"]
    date_candidates  = [
        "Transaction Date", "TransactionDate", "DeliveryDate",
        "Delivery Date", "Transaction Date Time", "TransactionDate Time"
    ]

    equip_col = next((c for c in equip_candidates if c in out.columns), None)
    date_col  = next((c for c in date_candidates  if c in out.columns), None)

    if equip_col is None or date_col is None:
        raise ValueError(
            "Kolom SO tidak ketemu. Pastikan ada kolom Equipment & tanggal delivery.\n"
            f"Kolom tersedia: {list(out.columns)}"
        )

    out = out.rename(columns={equip_col: "EquipmentNo", date_col: "DeliveryDate"}).copy()
    out["EquipmentNo"] = out["EquipmentNo"].astype(str).str.strip().str.upper()
    out["DeliveryDate"] = pd.to_datetime(out["DeliveryDate"], errors="coerce", dayfirst=True)
    out = out.dropna(subset=["EquipmentNo","DeliveryDate"]).copy()
    return out

### **UPLOAD DATA BULAN BARU (PKB & SO)**

*Mengunggah data PKB & SO terbaru tiap bulan (data dalam format excel)*

In [ ]:
from google.colab import files

if not os.path.exists(PKB_PATH):
    raise FileNotFoundError(f"Data PKB master belum ada di Drive: {PKB_PATH}")
pkb_master = clean_pkb(pd.read_excel(PKB_PATH))

if os.path.exists(SO_PATH):
    so_master = clean_so(pd.read_excel(SO_PATH))
else:
    so_master = pd.DataFrame(columns=["EquipmentNo","DeliveryDate"])

print("[OK] Master PKB:", pkb_master.shape, "| Master SO:", so_master.shape)

print("\nUPLOAD PKB bulan baru (xlsx)...")
uploaded_pkb = files.upload()
pkb_new_path = list(uploaded_pkb.keys())[0]
pkb_new = clean_pkb(pd.read_excel(pkb_new_path))

print("\nUPLOAD SO bulan baru (xlsx)...")
uploaded_so = files.upload()
so_new_path = list(uploaded_so.keys())[0]
so_new = clean_so(pd.read_excel(so_new_path))

# Dedup PKB by PKBNo
pkb_all = pd.concat([pkb_master, pkb_new], ignore_index=True)
pkb_all = (pkb_all.sort_values(["PKBDate","PKBNo"])
           .drop_duplicates(subset=["PKBNo"], keep="last")
           .reset_index(drop=True))

so_all = pd.concat([so_master, so_new], ignore_index=True)
so_all = (so_all.drop_duplicates(subset=["EquipmentNo","DeliveryDate"], keep="last")
          .reset_index(drop=True))

# Save updated history
pkb_all.to_excel(PKB_PATH, index=False)
so_all.to_excel(SO_PATH, index=False)

print("[OK] Updated PKB saved:", PKB_PATH, "| rows:", len(pkb_all))
print("[OK] Updated SO  saved:", SO_PATH,  "| rows:", len(so_all))

### **MERGE PKB + SO**

In [ ]:
so_one = (so_all.sort_values(["EquipmentNo","DeliveryDate"])
          .groupby("EquipmentNo", as_index=False)
          .first()[["EquipmentNo","DeliveryDate"]])

df = pkb_all.copy()
df["EquipmentNo"] = df["EquipmentNo"].astype(str).str.strip().str.upper()

for col in ["DeliveryDate", "delivered_here"]:
    if col in df.columns:
        df = df.drop(columns=[col])

df = df.merge(so_one, on="EquipmentNo", how="left")
df["delivered_here"] = df["DeliveryDate"].notna().astype(int)

### **FILTER DATA**

In [ ]:
tcare_cats    = ["SBI", "SBIK", "SBE", "SBEK", "EM", "EMK"]
regular_cats  = ["GR", "OILCHA"]
periodic_cats = tcare_cats + regular_cats

allowed_pkb_type = ["BOOKING", "FIELD SERVICE", "WALK IN", "FOLLOW UP FROM ANOTHER BRANCH"]

df["ServiceCategory"] = df["ServiceCategory"].astype(str).str.strip().str.upper()
df["PKBType"] = df["PKBType"].astype(str).str.strip().str.upper()

df = df[df["PKBType"].isin(allowed_pkb_type)].copy()
df = df[df["ServiceCategory"].isin(periodic_cats)].copy()
df = df.sort_values(["EquipmentNo","PKBDate","PKBNo"]).reset_index(drop=True)

AS_OF_DATE = df["PKBDate"].max().normalize()
RUN_DATE   = pd.Timestamp(f"{TARGET_MONTH}-01")

### **RULE**

In [ ]:
def end_of_month(dt):
    return pd.Timestamp(dt) + pd.offsets.MonthEnd(0)

def add_months_keep_dom(dt, months: int):
    dt = pd.Timestamp(dt)
    target = dt + pd.DateOffset(months=int(months))
    try:
        return target.replace(day=dt.day)
    except ValueError:
        return end_of_month(target)

def deadline_eom_from_target(target_date):
    """Deadline = akhir bulan dari target_date."""
    return end_of_month(pd.Timestamp(target_date))

# target schedule
def tcare_target_seq1(delivery_date):            # delivery + 1
    return add_months_keep_dom(delivery_date, 1)

def tcare_target_seq2(delivery_date):            # delivery + 6
    return add_months_keep_dom(delivery_date, 6)

def tcare_target_seq3_to_7(prev_service_date):   # prev + 6
    return add_months_keep_dom(prev_service_date, 6)

def regular_target_next(prev_service_date):      # prev + 6
    return add_months_keep_dom(prev_service_date, 6)

# deadline = EOM(target)
def tcare_deadline_seq1(delivery_date):
    return deadline_eom_from_target(tcare_target_seq1(delivery_date))

def tcare_deadline_seq2(delivery_date):
    return deadline_eom_from_target(tcare_target_seq2(delivery_date))

def tcare_deadline_seq3_to_7(prev_service_date):
    return deadline_eom_from_target(tcare_target_seq3_to_7(prev_service_date))

#### **RULE TCARE (DELIVERY DIKETAHUI)**

In [ ]:
tcare_hist = df[
    (df["ServiceCategory"].isin(tcare_cats)) &
    (df["DeliveryDate"].notna()) &
    (df["PKBDate"].notna()) &
    (df["PKBDate"] >= df["DeliveryDate"])
].copy()

tcare_hist = tcare_hist.sort_values(["EquipmentNo","PKBDate","PKBNo"]).copy()

print("\n=== TCARE HIST (KNOWN DELIVERY) ===")
print("Rows:", len(tcare_hist))
print("Unique vehicles:", tcare_hist["EquipmentNo"].nunique())

if len(tcare_hist) == 0:
    known_rule = pd.DataFrame(columns=[
        "EquipmentNo","DeliveryDate","LastTcareSeq","EverLateTcare","VehicleEligibleTcare","VehicleEligibleLabel",
        "NextReminderType","NextTargetDate","NextDeadlineTol","DaysToDeadline","ReasonRegular","RuleSource",
        "SnapshotPKBDate"
    ])
else:
    tcare_hist["TcareServiceSeq"] = tcare_hist.groupby("EquipmentNo").cumcount() + 1
    tcare_hist["PrevServiceDate"] = tcare_hist.groupby("EquipmentNo")["PKBDate"].shift(1)

    def compute_deadline_visit(row):
        seq = int(row["TcareServiceSeq"])
        if seq == 1:
            return tcare_deadline_seq1(row["DeliveryDate"])
        elif seq == 2:
            return tcare_deadline_seq2(row["DeliveryDate"])
        elif 3 <= seq <= 7:
            return tcare_deadline_seq3_to_7(row["PrevServiceDate"])
        return pd.NaT

    tcare_hist["DeadlineTolAtVisit"] = tcare_hist.apply(compute_deadline_visit, axis=1)

    tcare_hist["IsLateTcare"] = 0
    mask_prog = (tcare_hist["TcareServiceSeq"] <= 7)
    tcare_hist.loc[mask_prog, "IsLateTcare"] = (
        tcare_hist.loc[mask_prog, "PKBDate"] > tcare_hist.loc[mask_prog, "DeadlineTolAtVisit"]
    ).astype(int)

    tcare_hist["EverLateCount"] = tcare_hist.groupby("EquipmentNo")["IsLateTcare"].cumsum()
    tcare_hist["BenefitActiveAtVisit"] = (tcare_hist["EverLateCount"] == 0).astype(int)

    print("BenefitActiveAtVisit counts:\n", tcare_hist["BenefitActiveAtVisit"].value_counts(dropna=False))

    tcare_vehicle = (
        tcare_hist.groupby("EquipmentNo", as_index=False)
        .agg(
            DeliveryDate=("DeliveryDate", "first"),
            LastTcareSeq=("TcareServiceSeq", "max"),
            EverLateTcare=("IsLateTcare", "max"),
            VehicleEligibleTcare=("BenefitActiveAtVisit", "min")
        )
    )
    tcare_vehicle["VehicleEligibleLabel"] = np.where(
        tcare_vehicle["VehicleEligibleTcare"] == 1, "ELIGIBLE_TCARE", "NOT_ELIGIBLE_TCARE"
    )

    tcare_last_visit = (
        tcare_hist.sort_values(["EquipmentNo","PKBDate","PKBNo"])
        .groupby("EquipmentNo", as_index=False)
        .tail(1)
        .copy()
    )

    tcare_last_visit = tcare_last_visit.rename(columns={"PKBDate": "SnapshotPKBDate"})

    if "DeliveryDate" in tcare_last_visit.columns:
        tcare_last_visit = tcare_last_visit.drop(columns=["DeliveryDate"])

    tcare_last_visit = tcare_last_visit.merge(
        tcare_vehicle[[
            "EquipmentNo", "DeliveryDate",
            "LastTcareSeq", "EverLateTcare", "VehicleEligibleTcare", "VehicleEligibleLabel"
        ]],
        on="EquipmentNo",
        how="left",
        validate="one_to_one"
    )

    def decide_next_known(row):
        last_seq = int(row["LastTcareSeq"])
        eligible = int(row["VehicleEligibleTcare"])

        # jika sudah 7x / pernah telat => REGULAR
        if (last_seq >= 7) or (eligible == 0):
            next_type = "REGULAR"
            target = regular_target_next(row["SnapshotPKBDate"])
            deadline = deadline_eom_from_target(target)
            reason = "SELESAI_TCARE_7X" if last_seq >= 7 else "PERNAH_TELAT"
            return pd.Series([next_type, target, deadline, reason])

        # kalau masih eligible => next TCARE
        next_seq = last_seq + 1

        if next_seq == 1:
            target = tcare_target_seq1(row["DeliveryDate"])
            deadline = tcare_deadline_seq1(row["DeliveryDate"])
        elif next_seq == 2:
            target = tcare_target_seq2(row["DeliveryDate"])
            deadline = tcare_deadline_seq2(row["DeliveryDate"])
        elif 3 <= next_seq <= 7:
            target = tcare_target_seq3_to_7(row["SnapshotPKBDate"])
            deadline = tcare_deadline_seq3_to_7(row["SnapshotPKBDate"])
        else:
            target, deadline = pd.NaT, pd.NaT

        return pd.Series(["TCARE", target, deadline, None])

    tcare_last_visit[["NextReminderType","NextTargetDate","NextDeadlineTol","ReasonRegular"]] = (
        tcare_last_visit.apply(decide_next_known, axis=1)
    )

    tcare_last_visit["DaysToDeadline"] = (tcare_last_visit["NextDeadlineTol"] - RUN_DATE).dt.days
    tcare_last_visit["RuleSource"] = "KNOWN_DELIVERY_RULE"

    known_rule = tcare_last_visit[[
        "EquipmentNo","DeliveryDate",
        "LastTcareSeq","EverLateTcare","VehicleEligibleTcare","VehicleEligibleLabel",
        "NextReminderType","NextTargetDate","NextDeadlineTol","DaysToDeadline",
        "ReasonRegular","RuleSource",
        "SnapshotPKBDate"
    ]].copy()

*snapshot last visit all vehicles*

In [ ]:
snap_last = (
    df.sort_values(["EquipmentNo","PKBDate","PKBNo"])
      .groupby("EquipmentNo", as_index=False)
      .tail(1)
      .copy()
)

#### **RULE TCARE (DELIVERY TIDAK ADA)**

In [ ]:
unknown_rule = snap_last[
    (snap_last["DeliveryDate"].isna()) &
    (snap_last["ServiceCategory"].isin(tcare_cats))
].copy()

unknown_rule["SnapshotPKBDate"] = unknown_rule["PKBDate"]
unknown_rule["NextReminderType"] = "TCARE"
unknown_rule["NextTargetDate"]   = unknown_rule["SnapshotPKBDate"].apply(lambda d: add_months_keep_dom(d, 6))
unknown_rule["NextDeadlineTol"]  = unknown_rule["NextTargetDate"].apply(deadline_eom_from_target)
unknown_rule["DaysToDeadline"]   = (unknown_rule["NextDeadlineTol"] - RUN_DATE).dt.days

unknown_rule["LastTcareSeq"] = np.nan
unknown_rule["EverLateTcare"] = np.nan
unknown_rule["VehicleEligibleTcare"] = np.nan
unknown_rule["VehicleEligibleLabel"] = "UNKNOWN_DELIVERY"
unknown_rule["ReasonRegular"] = None
unknown_rule["RuleSource"] = "UNKNOWN_DELIVERY_PROXY"
unknown_rule["DeliveryDate"] = pd.NaT

unknown_rule = unknown_rule[known_rule.columns].copy() if len(known_rule.columns) else unknown_rule

#### **RULE REGULAR ( SERVIS BERKALA DILUAR TCARE)**

In [ ]:
regular_rule = snap_last[snap_last["ServiceCategory"].isin(regular_cats)].copy()
regular_rule["SnapshotPKBDate"] = regular_rule["PKBDate"]
regular_rule["NextReminderType"] = "REGULAR"
regular_rule["NextTargetDate"]   = regular_rule["SnapshotPKBDate"].apply(lambda d: add_months_keep_dom(d, 6))
regular_rule["NextDeadlineTol"]  = regular_rule["NextTargetDate"].apply(deadline_eom_from_target)
regular_rule["DaysToDeadline"]   = (regular_rule["NextDeadlineTol"] - RUN_DATE).dt.days

regular_rule["LastTcareSeq"] = np.nan
regular_rule["EverLateTcare"] = np.nan
regular_rule["VehicleEligibleTcare"] = np.nan
regular_rule["VehicleEligibleLabel"] = "REGULAR_SERVICE"
regular_rule["ReasonRegular"] = "REGULAR_SERVICE"
regular_rule["RuleSource"] = "REGULAR_SERVICE"
regular_rule["DeliveryDate"] = regular_rule.get("DeliveryDate", pd.NaT)

regular_rule = regular_rule[known_rule.columns].copy() if len(known_rule.columns) else regular_rule

#### **FINAL RULE**

In [ ]:
tcare_covered = set(pd.concat([known_rule["EquipmentNo"], unknown_rule["EquipmentNo"]]).unique())
regular_rule = regular_rule[~regular_rule["EquipmentNo"].isin(tcare_covered)].copy()

final_rule = pd.concat([known_rule, unknown_rule, regular_rule], ignore_index=True)

final_rule["NextTargetDate"]  = pd.to_datetime(final_rule["NextTargetDate"], errors="coerce")
final_rule["NextDeadlineTol"] = pd.to_datetime(final_rule["NextDeadlineTol"], errors="coerce")

final_rule["UrgencyBucket"] = pd.cut(
    final_rule["DaysToDeadline"],
    bins=[-np.inf, 0, 7, 14, 30, np.inf],
    labels=["OVERDUE","DUE_7D","DUE_14D","DUE_30D","FAR"],
    right=True
).astype("string").fillna("FAR")

final_rule["ReminderMonth"] = final_rule["NextTargetDate"].dt.to_period("M").astype(str)
final_rule["ReminderMonthDate"] = final_rule["NextTargetDate"].dt.to_period("M").dt.to_timestamp()

priority_order = {"TCARE": 0, "REGULAR": 1}
final_ranked = (
    final_rule.assign(_p = final_rule["NextReminderType"].map(priority_order).fillna(9).astype(int))
             .sort_values(["_p", "DaysToDeadline"], ascending=[True, True])
             .drop(columns=["_p"])
             .reset_index(drop=True)
)

print("\n=== FINAL RULE CHECK ===")
print("Rows:", len(final_ranked), "| uniq vehicles:", final_ranked["EquipmentNo"].nunique())
print(final_ranked["NextReminderType"].value_counts(dropna=False))
print(final_ranked["UrgencyBucket"].value_counts(dropna=False))

### **MODEL**

In [ ]:
base = df.sort_values(["EquipmentNo","PKBDate","PKBNo"]).copy()
base["gap_days"] = base.groupby("EquipmentNo")["PKBDate"].diff().dt.days

gap_ops = (base.groupby("EquipmentNo", as_index=False)
           .agg(avg_gap_days=("gap_days","mean"),
                last_gap_days=("gap_days","last")))

win_ops = base[(base["PKBDate"] > AS_OF_DATE - pd.DateOffset(months=12)) & (base["PKBDate"] <= AS_OF_DATE)]
freq_ops = (win_ops.groupby("EquipmentNo", as_index=False)
            .agg(visit_count_12m=("PKBDate","count")))

id_candidates = ["EquipmentNo","PoliceRegNo","CustomerName","ContactPerson","ContactPersonPhone"]
extra_candidates = ["ServiceCategory","PKBType","TypeKendaraan","ServiceAdvisor","Kilometer","EstimatedRevenue"]

id_exist = [c for c in id_candidates if c in snap_last.columns]
extra_exist = [c for c in extra_candidates if c in snap_last.columns]

ops = final_ranked.merge(
    snap_last[id_exist + extra_exist],
    on="EquipmentNo",
    how="left",
    validate="one_to_one"
)

ops = ops.merge(gap_ops, on="EquipmentNo", how="left")
ops = ops.merge(freq_ops, on="EquipmentNo", how="left")

# fitur numeric operasional
ops["recency_days"] = (AS_OF_DATE - pd.to_datetime(ops.get("SnapshotPKBDate", pd.NaT), errors="coerce")).dt.days
ops["days_to_deadline"] = ops["DaysToDeadline"]

ops["last_km"] = pd.to_numeric(ops.get("Kilometer", np.nan), errors="coerce")
ops["last_revenue"] = pd.to_numeric(ops.get("EstimatedRevenue", np.nan), errors="coerce")

for c in ["avg_gap_days","last_gap_days","visit_count_12m","recency_days","days_to_deadline","last_km","last_revenue"]:
    if c in ops.columns:
        ops[c] = pd.to_numeric(ops[c], errors="coerce").fillna(0)

# kategorikal
for c in cat_feats:
    if c not in ops.columns:
        ops[c] = "UNKNOWN"
    ops[c] = ops[c].fillna("UNKNOWN").astype(str).str.strip().str.upper()

for c in num_feats:
    if c not in ops.columns:
        ops[c] = 0.0

X_ops = ops[num_feats + cat_feats].copy()

# scoring: proba_on_time => risk_score = 1 - proba
proba_on_time = pipe_xgb.predict_proba(X_ops)[:, 1]
ops["proba_on_time"] = proba_on_time
ops["risk_score"] = 1 - proba_on_time

ops_final = (
    ops.assign(_p = ops["NextReminderType"].map(priority_order).fillna(9).astype(int))
       .sort_values(["_p", "DaysToDeadline", "risk_score"], ascending=[True, True, False])
       .drop(columns=["_p"])
       .reset_index(drop=True)
)

print("[OK] ops_final:", ops_final.shape)

#### **FINAL SCORING LIST REMINDER**

In [ ]:
ops_final["Bulan"] = ops_final["ReminderMonth"]
ops_final["fact_key"] = ops_final["EquipmentNo"].astype(str) + "_" + ops_final["Bulan"].astype(str)

if "status_reminder" not in ops_final.columns:
    ops_final["status_reminder"] = "BELUM"

fact_cols = [
    "Bulan","EquipmentNo","PoliceRegNo","CustomerName",
    "ServiceCategory","PKBType","TypeKendaraan","ServiceAdvisor",
    "SnapshotPKBDate","DeliveryDate",
    "NextReminderType","NextTargetDate","NextDeadlineTol","DaysToDeadline",
    "UrgencyBucket","RuleSource", "Kilometer", "EstimatedRevenue",
    "proba_on_time","risk_score",
    "status_reminder","fact_key"
]
fact_cols = [c for c in fact_cols if c in ops_final.columns]
fact_df = ops_final[fact_cols].copy()

# Reminder_Target: hanya bulan target + urgent (DUE_7/14/30)
active_df = ops_final[
    (ops_final["Bulan"] == TARGET_MONTH) &
    (ops_final["NextTargetDate"].notna()) &
    (ops_final["UrgencyBucket"].isin(["DUE_7D","DUE_14D","DUE_30D"]))
].copy()

### **INTEGRASI GOOGLE SHEET (WA LINK & MONITORING POWER BI)**

In [ ]:
import re, json
import numpy as np
import pandas as pd
from urllib.parse import quote

def normalize_phone(phone):
    if pd.isna(phone):
        return None

    s = str(phone).strip()
    s = re.split(r"[;/,]| atau | and |&|\n", s, flags=re.IGNORECASE)[0].strip()
    s = re.split(r"(ext\.?|extension|x|#|samb\.?|sambungan)\s*\d+", s, flags=re.IGNORECASE)[0].strip()

    digits = re.sub(r"\D", "", s)
    if not digits:
        return None

    if digits.startswith("00"):
        digits = digits[2:]

    # normalisasi
    if digits.startswith("0"):
        digits = "62" + digits[1:]
    elif digits.startswith("8"):
        digits = "62" + digits
    elif digits.startswith("62"):
        pass
    else:
        return None

    if digits.startswith("6208"):
        digits = "628" + digits[4:]
    elif digits.startswith("620"):
        digits = "62" + digits[3:]

    digits = re.sub(r"^62+0+", "62", digits)

    # validasi panjang E.164 (10-15 digit)
    if len(digits) < 10 or len(digits) > 15:
        return None
    if not digits.startswith("628"):
        return None

    return digits


def pick_first_nonempty(*vals, default=""):
    for v in vals:
        if pd.notna(v):
            s = str(v).strip()
            if s and s.upper() != "UNKNOWN":
                return s
    return default


def build_message(row):
    name  = pick_first_nonempty(row.get("CustomerName"), row.get("ContactPerson"), default="AutoFamily")
    plate = pick_first_nonempty(row.get("PoliceRegNo"), default="(plat belum tersedia)")
    car   = pick_first_nonempty(row.get("TypeKendaraan"), default="Toyota")

    due = row.get("NextDeadlineTol", pd.NaT)
    if pd.isna(due):
        due = row.get("NextTargetDate", pd.NaT)

    try:
        due_txt = pd.to_datetime(due).strftime("%d %b %Y") if pd.notna(due) else "-"
    except Exception:
        due_txt = "-"

    is_tcare = str(row.get("NextReminderType","")).upper() == "TCARE"

    if is_tcare:
        return f"""Selamat siang Bapak/Ibu {name},
Kiranya sehat selalu.

Kami menginformasikan untuk kendaraan {car} {plate} sudah waktunya servis berkala.
___
Kendaraan AutoFamily mendapatkan Gratis Oli Mesin, Filter Oli dan Gasket
serta Gratis Biaya Jasa Service Berkala.
___
Lakukan servis berkala sesuai jadwal dan dapatkan benefit Program T-Care:
Free biaya Jasa & Part (Maks 3 tahun / 60.000 Km),
Serta Extended Warranty (1 tahun / 20.000 Km),
*Sesuai syarat & ketentuan program.
https://aftersales.toyota.astra.co.id/t-care
___
Segera lakukan Booking Service dengan balas Ya (min H-2).

Salam hangat,
PIC Booking Service
Authorized Service Branch.
WhatsApp: [BRANCH_CONTACT]
Hotline: [BRANCH_HOTLINE]
[BRANCH_ADDRESS]"""
    else:
        return f"""Selamat siang Bapak/Ibu {name},
Kiranya sehat selalu.

Kami menginformasikan untuk kendaraan {car} {plate} sudah waktunya servis berkala (perkiraan jatuh tempo: {due_txt}).
___
Segera lakukan Booking Service dengan balas Ya (min H-2).

Salam hangat,
PIC Booking Service
Authorized Service Branch.
WhatsApp: [BRANCH_CONTACT]
Hotline: [BRANCH_HOTLINE]
[BRANCH_ADDRESS]"""


def build_wa_link(phone, message):
    ph = normalize_phone(phone)
    if not ph:
        return ""
    return f"https://wa.me/{ph}?text={quote(message)}"


def phone_reason(raw):
    """Buat audit kenapa link bisa kosong."""
    if pd.isna(raw) or str(raw).strip() == "" or str(raw).strip().lower() == "nan":
        return "EMPTY"

    s = str(raw).strip()
    s = re.split(r"[;/,]| atau | and |&|\n", s, flags=re.IGNORECASE)[0].strip()
    s = re.split(r"(ext\.?|extension|x|#|samb\.?|sambungan)\s*\d+", s, flags=re.IGNORECASE)[0].strip()
    d = re.sub(r"\D", "", s)
    if not d:
        return "NO_DIGITS"

    ph = normalize_phone(raw)
    if ph:
        return "OK"

    if d.startswith("0") and len(d) >= 3 and d[1] in list("234567"):
        return "LIKELY_LANDLINE"

    return "INVALID_FORMAT"

In [ ]:
assert "active_df" in globals(), "active_df belum ada."

active_df = active_df.copy()

if "status_reminder" not in active_df.columns:
    active_df["status_reminder"] = "BELUM"

active_df["wa_text"] = active_df.apply(build_message, axis=1)

phone_col_candidates = [
    "ContactPersonPhone",
    "Mobile Phone",
    "MobilePhone",
    "Phone",
    "NoHP"
]
phone_col = next((c for c in phone_col_candidates if c in active_df.columns), None)
print("[CHECK] phone_col dipakai:", phone_col)

if phone_col is None:
    active_df["wa_link"] = ""
    active_df["phone_norm"] = ""
    active_df["phone_reason"] = "NO_PHONE_COLUMN"
else:
    active_df["phone_norm"] = active_df[phone_col].apply(normalize_phone)
    active_df["phone_reason"] = active_df[phone_col].apply(phone_reason)

    active_df["wa_link"] = active_df.apply(
        lambda r: build_wa_link(r.get(phone_col, np.nan), r["wa_text"]),
        axis=1
    )

print("[CHECK] wa_link non-empty:", (active_df["wa_link"].astype(str).str.len() > 0).sum())
print("[CHECK] phone_reason counts:\n", active_df["phone_reason"].value_counts(dropna=False).head(10))

In [ ]:

import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth
from google.auth import default
from pandas.api.types import is_datetime64_any_dtype, is_period_dtype, is_timedelta64_dtype

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

def open_or_create_spreadsheet(client: gspread.Client, name: str) -> gspread.Spreadsheet:
    return client.open(name)

def open_or_create_worksheet(sh: gspread.Spreadsheet, title: str, rows: int, cols: int) -> gspread.Worksheet:
    try:
        ws = sh.worksheet(title)
        ws.resize(rows=max(rows, 100), cols=max(cols, 10))
        return ws
    except gspread.WorksheetNotFound:
        return sh.add_worksheet(title=title, rows=max(rows, 100), cols=max(cols, 10))

def _to_json_if_needed(x):
    if isinstance(x, (list, dict)):
        return json.dumps(x, ensure_ascii=False)
    return x

def write_df_values_only(ws: gspread.Worksheet, df: pd.DataFrame):
    ws.batch_clear(["A:ZZ"])

    out = df.copy()
    for c in out.columns:
        s = out[c]
        if is_datetime64_any_dtype(s):
            out[c] = pd.to_datetime(s, errors="coerce").dt.strftime("%Y-%m-%d")
        elif is_period_dtype(s) or is_timedelta64_dtype(s):
            out[c] = s.astype(str)
        else:
            out[c] = s.map(_to_json_if_needed)

    out = out.replace({np.nan: ""}).astype(object)
    set_with_dataframe(ws, out, include_index=False, resize=True)

def read_ws_as_df(sh: gspread.Spreadsheet, sheet_name: str) -> pd.DataFrame:
    try:
        ws = sh.worksheet(sheet_name)
        values = ws.get_all_values()
        if not values or len(values) < 2:
            return pd.DataFrame()
        header = values[0]
        rows = values[1:]
        return pd.DataFrame(rows, columns=header)
    except Exception:
        return pd.DataFrame()

def clear_data_validation(ws: gspread.Worksheet, start_row=2, max_rows=6000, max_cols=26):
    req = [{
        "setDataValidation": {
            "range": {
                "sheetId": ws.id,
                "startRowIndex": start_row - 1,
                "endRowIndex": max_rows,
                "startColumnIndex": 0,
                "endColumnIndex": max_cols
            },
            "rule": None
        }
    }]
    ws.spreadsheet.batch_update({"requests": req})

def set_status_dropdown(ws: gspread.Worksheet,
                        col_name="status_reminder", allowed=("BELUM", "SUDAH"),
                        start_row=2, max_rows=5000):
    headers = ws.row_values(1)
    headers_norm = [h.strip().lower() for h in headers]
    key = col_name.strip().lower()
    if key not in headers_norm:
        print(f"[INFO] Kolom '{col_name}' tidak ditemukan, dropdown dilewati.")
        return

    col_idx = headers_norm.index(key)
    requests = [{
        "setDataValidation": {
            "range": {
                "sheetId": ws.id,
                "startRowIndex": start_row - 1,
                "endRowIndex": max_rows,
                "startColumnIndex": col_idx,
                "endColumnIndex": col_idx + 1
            },
            "rule": {
                "condition": {
                    "type": "ONE_OF_LIST",
                    "values": [{"userEnteredValue": v} for v in allowed]
                },
                "strict": True,
                "showCustomUi": True
            }
        }
    }]
    ws.spreadsheet.batch_update({"requests": requests})

def hide_column_by_name(ws: gspread.Worksheet, col_name: str):
    headers = ws.row_values(1)
    headers_norm = [h.strip().lower() for h in headers]
    key = col_name.strip().lower()
    if key not in headers_norm:
        return
    col_idx_1based = headers_norm.index(key) + 1

    req = [{
        "updateDimensionProperties": {
            "range": {
                "sheetId": ws.id,
                "dimension": "COLUMNS",
                "startIndex": col_idx_1based - 1,
                "endIndex": col_idx_1based
            },
            "properties": {"hiddenByUser": True},
            "fields": "hiddenByUser"
        }
    }]
    ws.spreadsheet.batch_update({"requests": req})

def unhide_column_by_name(ws: gspread.Worksheet, col_name: str):
    headers = ws.row_values(1)
    headers_norm = [h.strip().lower() for h in headers]
    key = col_name.strip().lower()
    if key not in headers_norm:
        return
    col_idx_1based = headers_norm.index(key) + 1

    req = [{
        "updateDimensionProperties": {
            "range": {
                "sheetId": ws.id,
                "dimension": "COLUMNS",
                "startIndex": col_idx_1based - 1,
                "endIndex": col_idx_1based
            },
            "properties": {"hiddenByUser": False},
            "fields": "hiddenByUser"
        }
    }]
    ws.spreadsheet.batch_update({"requests": req})


# open
sh = open_or_create_spreadsheet(gc, GSHEET_NAME)

# read old active untuk preserve status_reminder
old_active = read_ws_as_df(sh, WS_ACTIVE)
if not old_active.empty:
    old_active.columns = [c.strip() for c in old_active.columns]

active_new = active_df.copy()
active_new.columns = [c.strip() for c in active_new.columns]

if "fact_key" not in active_new.columns:
    active_new["fact_key"] = active_new["EquipmentNo"].astype(str) + "_" + active_new["Bulan"].astype(str)
active_new["fact_key"] = active_new["fact_key"].astype(str)

if "status_reminder" not in active_new.columns:
    active_new["status_reminder"] = "BELUM"

# preserve status dari sheet lama
if (not old_active.empty) and {"fact_key","status_reminder"}.issubset(old_active.columns):
    keep = old_active[["fact_key","status_reminder"]].copy()
    keep["fact_key"] = keep["fact_key"].astype(str)

    active_new = active_new.merge(keep, on="fact_key", how="left", suffixes=("", "_old"))
    active_new["status_reminder"] = (
        active_new["status_reminder_old"].replace("", np.nan).fillna(active_new["status_reminder"])
    )
    active_new = active_new.drop(columns=["status_reminder_old"])

# sanitize status_reminder
active_new["status_reminder"] = (
    active_new["status_reminder"].astype(str).str.strip().str.upper()
    .replace({"NAN": "BELUM", "": "BELUM"})
)
active_new.loc[~active_new["status_reminder"].isin(["BELUM","SUDAH"]), "status_reminder"] = "BELUM"

# Reminder_Target columns (tambahin audit cols biar gampang cek)
oper_cols = [
    "Bulan","CustomerName","ContactPerson","ContactPersonPhone","PoliceRegNo","EquipmentNo",
    "TypeKendaraan","ServiceCategory","ServiceAdvisor","UrgencyBucket",
    "status_reminder","wa_link",
    "phone_norm", "fact_key"
]
oper_cols = [c for c in oper_cols if c in active_new.columns]
active_out = active_new[oper_cols].copy()

print("[DEBUG] active_out columns:", list(active_out.columns))
assert "wa_link" in active_out.columns, "wa_link tidak ikut ke active_out."

ws_fact = open_or_create_worksheet(sh, WS_FACT, rows=len(fact_df) + 10, cols=len(fact_df.columns) + 5)
ws_act  = open_or_create_worksheet(sh, WS_ACTIVE, rows=len(active_out) + 10, cols=len(active_out.columns) + 5)

write_df_values_only(ws_fact, fact_df)
write_df_values_only(ws_act, active_out)

clear_data_validation(ws_act, start_row=2, max_rows=max(6000, len(active_out) + 200), max_cols=30)

set_status_dropdown(ws_act, col_name="status_reminder",
                    allowed=("BELUM","SUDAH"),
                    start_row=2, max_rows=max(6000, len(active_out) + 200))

hide_column_by_name(ws_act, "fact_key")
unhide_column_by_name(ws_act, "wa_link")

#### **SELESAI**

In [ ]:
print("[OK] Google Sheet updated:", sh.url)
print(" - All Customer rows    :", len(fact_df))
print(" - Reminder_Target rows :", len(active_out))